# ScratchFormer: Transformer from Scratch (Colab TPU / GPU Runner)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikishore710/ScratchFormer/blob/main/notebooks/ScratchFormer_Colab_TPU_GPU.ipynb)

A complete from-scratch encoder-decoder Transformer in TensorFlow/Keras for English → French machine translation.
- **Pure First Principles**: Custom multi-head attention, scaled dot-product attention, sinusoidal positional encoding, padding & causal masks, and residual post-norm blocks. No 	f.keras.layers.MultiHeadAttention in the main model.
- **Hardware-Accelerated**: Designed for Google Colab **T4 GPU** or **TPU v2/v3**.
- **Self-Contained Execution**: Runs tests, smoke gates, full 10-epoch training, test evaluation, attention heatmaps, and the 14-run ablation study.



## 1. Accelerator & Environment Setup
Select runtime: **Runtime → Change runtime type → T4 GPU** (or TPU).


In [ ]:
import os
import sys
import tensorflow as tf

# Accelerator detection
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow Version: {tf.__version__}')

tpu_strategy = None
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    tpu_strategy = tf.distribute.TPUStrategy(resolver)
    print(f'Running on TPU with {tpu_strategy.num_replicas_in_sync} replicas')
except (ValueError, tf.errors.NotFoundError):
    if gpus:
        print(f'Running on GPU: {[gpu.name for gpu in gpus]}')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print('Running on CPU')

# Clone repository if running in Colab
if 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.exists('ScratchFormer'):
        !git clone https://github.com/Ravikishore710/ScratchFormer.git
    %cd ScratchFormer
    !pip install -q -r requirements.txt

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

print('Current Working Directory:', os.getcwd())



## 2. Advanced Test Suite Verification (31 Unit & Integration Tests)
Runs all unit and integration tests covering masks, scaled dot-product attention, layer parity, tokenizers, teacher-forcing alignment, inference, and the end-to-end pipeline.


In [ ]:
!python -m pytest tests/ -v --tb=short


## 3. Fast Pipeline Smoke Test
Validates the 4 smoke stages (data pipeline, 1-epoch overfit check gate, greedy decoding, and attention heatmaps) before starting long training.


In [ ]:
# 3a. Data pipeline validation
from src.config import Config
from src.data.dataset import prepare_datasets
b_smoke = prepare_datasets(Config(max_pairs=2000), verbose=True)
print('Train batch src shape:', next(iter(b_smoke.train_ds.take(1)))[0][0].shape)

# 3b. Model + Overfit Check + 1 Epoch smoke run
from src.model.transformer import Transformer, count_parameters
from src.training.trainer import Trainer

cfg_smoke = Config(max_pairs=2000, epochs=1, batch_size=64)
m_smoke = Transformer(cfg_smoke, len(b_smoke.src_tok), len(b_smoke.tgt_tok))
print('Smoke Model Parameters:', count_parameters(m_smoke))
t_smoke = Trainer(m_smoke, cfg_smoke, checkpoint_dir='outputs/model/smoke')
t_smoke.overfit_check(b_smoke.train_ds, steps=50)
h_smoke = t_smoke.fit(b_smoke.train_ds, b_smoke.val_ds, log_every=10)
print('Smoke history:', h_smoke)

# 3c. Greedy translation smoke
from src.inference.generate import translate
t_smoke.restore_latest()
print('Sample Translation:', translate(m_smoke, b_smoke.src_tok, b_smoke.tgt_tok, b_smoke.test_src_text[0]))

# 3d. Clean up smoke checkpoint
import shutil
shutil.rmtree('outputs/model/smoke', ignore_errors=True)
print('Smoke tests passed successfully!')



## 4. Full Baseline Training (60,000 Sentence Pairs, 10 Epochs)
Trains the baseline ScratchFormer model ({model}=128, h=8, L=2, d_{ff}=512$) on 60k pairs with learning-rate warmup and best-val checkpointing.


In [ ]:
!python scripts/train.py --config configs/baseline.json --epochs 10


### Display Training Curves (Loss & Token Accuracy)


In [ ]:
from IPython.display import Image, display
display(Image('outputs/training_curves/loss.png'))
display(Image('outputs/training_curves/acc.png'))



## 5. Held-Out Test Evaluation & Sample Predictions
Evaluates BLEU score, exact match percentage, and generation lengths on 500 held-out test sentences.


In [ ]:
!python scripts/evaluate.py --config configs/baseline.json --n 500


In [ ]:
import json
import pandas as pd

with open('outputs/predictions/test_metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

df_preds = pd.read_csv('outputs/predictions/test_predictions.csv')
print('\n=== Sample Test Predictions (Top 10) ===')
display(df_preds[['source', 'reference', 'hypothesis', 'exact_match']].head(10))



## 6. Attention Heatmap Visualizations
Visualizes encoder self-attention, causal decoder self-attention, and encoder-decoder cross-attention heatmaps.


In [ ]:
!python scripts/visualize.py --config configs/baseline.json --n 3


In [ ]:
import glob
from IPython.display import Image, display

print('--- Positional Encoding ---')
display(Image('outputs/positional_encoding/positional_encoding.png'))

print('--- Encoder Self-Attention (Test Example 0, Layer 0) ---')
display(Image('outputs/attention_maps/encoder/test_0_layer0.png'))

print('--- Decoder Causal Self-Attention (Test Example 0, Layer 0) ---')
display(Image('outputs/attention_maps/decoder_self/test_0_layer0.png'))

print('--- Cross-Attention (Test Example 0, Layer 0) ---')
display(Image('outputs/attention_maps/cross/test_0_layer0.png'))



## 7. Ablation Grid & Keras-MHA Benchmark
Runs all 13 ablation experiments plus the 	f.keras.layers.MultiHeadAttention benchmark row to measure the impact of:
- Positional encoding (on vs. off)
- Number of attention heads (1, 2, 4, 8)
- Model width {model}$ (64, 128, 256)
- Depth $ (1, 2, 4 layers)
- Feed-forward capacity {ff}$ (256, 512, 1024)
- Warmup duration (1000, 4000, 8000 steps)
- Built-in framework MHA vs. custom implementation



In [ ]:
!python scripts/run_experiments.py --config configs/baseline.json --epochs 3


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_exp = pd.read_csv('outputs/experiments/experiment_results.csv')
print('=== Full Ablation Experiment Table ===')
display(df_exp)

# Size vs. Quality Trade-off Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_exp['params'], df_exp['val_bleu'], c='royalblue', s=80, edgecolors='black', alpha=0.8)
for _, r in df_exp.iterrows():
    ax.annotate(r['experiment'], (r['params'], r['val_bleu']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('Trainable Parameters')
ax.set_ylabel('Validation BLEU')
ax.set_title('Size vs. Quality Trade-off across Ablations')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()



## 8. Export Results & Package Artifacts
Zips all generated results, logs, model weights, and figures for easy download.


In [ ]:
!zip -r scratchformer_results.zip outputs/
from google.colab import files
files.download('scratchformer_results.zip')
print('Results package ready!')

